# Solución Ejercicio 04b  |  Tema 4 - PRyES
## Estimación por máxima verosimilitud con ayuda de chatGPT

stats

In [ ]:
# import required packages
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
import pandas as pd
from scipy import optimize as opt  # para estimación por máxima verosimilitud

### Se sabe que la variable aleatoria X se distribuye como una uniforme entre 0 y b, es decir X ~U(0, b). Se quiere estimar el parámetro b de la distribución teórica, para lo que se dispone de la muestra de datos almacenada en muestra_Unif_0_b.csv

In [ ]:
# Leemos los datos de la muestra
x = np.loadtxt('muestra_2000_Unif_0_b.csv', delimiter=',', skiprows=1)

# tamaño de la muestra
n = len(x)
print(f'La muestra tiene {n} observaciones.')

### a) Representa el histograma de la muestra de datos disponible. ¿Se parece a una uniforme?

In [ ]:
# Respuesta

nbins = 20 # número de intervalos, ajustar para buena visualización

fig = plt.figure(1, figsize=(8, 4)) # permite indicar el nº de la figura y las dimensiones (ancho y alto)
sns.histplot(data=x, bins=nbins)
sns.rugplot(x, color='r')  # opcional
plt.title(f'Histograma ({nbins} intervalos)')
plt.xlabel('x')
plt.ylabel('Frecuencia absoluta')
plt.show()

print('Sí parece una distribución uniforme')

### b) Estima de manera sencilla, utilizando tu intuición, a partir de la muestra de 2000 observaciones, el valor b de la distribución Uniforme de X~U(0, b)

In [ ]:
# Respuesta

max_x = np.max(x)
b_est = max_x
print('De manera intuitiva, parece sensato pensar que el valor máximo observado sea el valor b')
print(f'Podemos estimar entonces que b = max(x) = {b_est}, por lo que X~U(0, {b_est})')

### c) Estima por máxima verosimilitud, a partir de la muestra de 2000 observaciones, el valor b de la distribución Uniforme de X~U(0, b). Utiliza como punto inicial de búsqueda el valor 6. Prueba ahora con 7. ¿Y si empiezas en 5?¿Cambia el valor estimado?¿Converge?

In [ ]:
# Respuesta
# define la función de verosimilitud logarítmica para maximizarla

def verosimilitud_log(parameters):
    # sacamos los parámetros
    param1 = parameters
    
    # La función de verosimilitud logarítmica depende del problema y requiere
    # conocer la distribución de la v.a. bajo estudio, que dependerá de los params
    # a estimar por MV
    
    inicio = 0
    incremento = param1 - 0
    VA = stats.uniform(loc=inicio, scale=incremento)
    logLikelihood = np.sum(stats.uniform.logpdf(x, loc=inicio, scale=incremento))  # <--- logpdf es el log de la pdf, ya está preparado ...
        
    # Como la función minimiza y queremos los params que maximizan la verosimilitud
    # simplemente la multimplicamos por -1: max(loglike) = min(-loglike)
    neg_logLikelihood = -1 * logLikelihood
    return neg_logLikelihood 

In [ ]:
# Maximiza la función de verosimilitud logarítmica (minimizando -1 * funcion)
# usamos como punto inicial de búsqueda
x0 = 5
mle_model = opt.minimize(verosimilitud_log, np.array(x0), method='L-BFGS-B')

# cogemos los coefs estimados
param1 = mle_model.x[0]

print(mle_model)
print('Parámetros estimados por MV:\n'
      f'  - parámetro 1: {param1}')


print('\nNo parece que esté funcionando muy bien la estimación. Hay problemas de convergencia y la estimación de b '
      'obtenida parece ser siempre igual al valor inicial de búsqueda.')

### d) Pregunta a chatGPT cómo se obtiene el límite superior de una uniforme. ¿Lo entiendes?

In [ ]:
# Respuesta

print('Preguntamos lo siguiente: "how can i estimate the upper limit of a uniform distribution by maximum likelihood?"')

In [ ]:
# Respuesta de chatGPT

print('Respuesta de chatGPT:')
print('''
To estimate the upper limit of a uniform distribution by maximum likelihood, you can follow these steps:

Write down the likelihood function for a uniform distribution with unknown upper limit, which is the 
probability of observing the data given the unknown parameter. For a sample of n observations x1, x2, ..., xn, the 
likelihood function is:

L(θ | x1, x2, ..., xn) = 1/(θ^n) if x1, x2, ..., xn are all less than or equal to θ, and 0 otherwise,

where θ is the upper limit of the uniform distribution.

Take the natural logarithm of the likelihood function to obtain the log-likelihood:

ln L(θ | x1, x2, ..., xn) = -n ln θ if x1, x2, ..., xn are all less than or equal to θ, and -∞ otherwise.

Find the value of θ that maximizes the log-likelihood function by taking the derivative of the
log-likelihood function with respect to θ and setting it equal to zero:

d/dθ ln L(θ | x1, x2, ..., xn) = -n/θ if x1, x2, ..., xn are all less than or equal to θ, and undefined otherwise.

The derivative is undefined when any of the observations is greater 
than the upper limit, because in that case the likelihood is zero and the 
log-likelihood is negative infinity. Therefore, we must restrict the optimization 
to the set of possible values of θ that are greater than or equal to the largest observation, 
denoted by xmax. Thus, the maximizer of the log-likelihood is:

^θ = xmax, which is the upper limit estimate by maximum likelihood.

To assess the uncertainty in the estimate, you can compute the standard error of the maximum 
likelihood estimator, which is:

se(^θ) = 0 if n = 1, and (n/(n-1))^(1/2) (xmax - xbar) if n > 1,

where xbar is the sample mean. The standard error is zero when there is 
only one observation, because there is no variation in the data. Otherwise,
the standard error is proportional to the sample range (xmax - xmin) and the square root of the sample size.

Note that this method assumes that the data are independent and identically distributed (iid)
and that the lower limit of the uniform distribution is known to be zero. If the lower limit 
is also unknown, the maximum likelihood estimator of the upper limit is given by ^θ = xmax + (xmax - xmin)/n. 
If the data are not uniformly distributed, the maximum likelihood estimator may not be the best estimator, 
and other methods may be more appropriate.

''')

### e) En la respuesta anterior de ChatGPT, identifica por qué la estimación numérica llamando a "opt.minimize" no está funcionando correctamente.

In [ ]:
# Respuesta

print('La derivada no está definida si hay un observación mayor que el parámetro b, ya que tendrá densidad nula y log(0) es -inf.'
      'Este es el detalle:')

print('''
The derivative is undefined when any of the observations is greater 
than the upper limit, because in that case the likelihood is zero and the 
log-likelihood is negative infinity. Therefore, we must restrict the optimization 
to the set of possible values of θ that are greater than or equal to the largest observation, 
denoted by xmax.
''')

### f) [OPCIONAL] Sigue la sugerencia de ChatGPT y plantea una optimización específica para este caso.

In [ ]:
# Respuesta

print('Parece entonces que el problema es que durante la búsqueda del mínimo '
      f'no debe pasar por ninguna razón que el parámetro sea menor que {b_est}')

print('Pedimos ayuda a chatGPT: "give me an example in python using scipy.optimize.minimize to set a constraint in the limit"')

print('chatGPT:: >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>')
      
print('''
Sure, here's an example of using scipy.optimize.minimize to find the maximum 
likelihood estimate of the upper limit of a uniform distribution subject to the 
constraint that the estimate must be greater than or equal to the largest observation in the sample.

import numpy as np
from scipy.optimize import minimize

# generate some sample data from a uniform distribution
np.random.seed(123)
n = 50
x = np.random.uniform(low=0, high=10, size=n)

# define the negative log-likelihood function for a uniform distribution
def neg_log_likelihood(theta, x):
    if all(xi <= theta for xi in x):
        return -n * np.log(theta)
    else:
        return np.inf

# define the constraint that the upper limit estimate must be greater than or equal to xmax
def constraint(theta, x):
    return theta - np.max(x)

# set initial guess for theta
theta0 = 5.0

# minimize the negative log-likelihood subject to the constraint using the SLSQP method
result = minimize(neg_log_likelihood, theta0, args=(x,), method='SLSQP', 
                  constraints=[{'type': 'ineq', 'fun': constraint, 'args': (x,)}])

# print the results
print(result)
In this example, we first generate a sample of 50 observations from 
a uniform distribution with lower limit 0 and upper limit 10. We define
the negative log-likelihood function for a uniform distribution and the constraint
that the upper limit estimate must be greater than or equal to the largest observation
in the sample. We set an initial guess for the upper limit estimate, and then we use
scipy.optimize.minimize to minimize the negative log-likelihood subject to the constraint using the SLSQP method.

The result object contains information about the optimization, including the 
estimate of the upper limit (result.x) and the value of the negative log-likelihood at
the estimate (result.fun). The estimate of the upper limit should be greater than or equal 
to the largest observation in the sample. If the constraint is violated, an error will be raised.

''')

In [ ]:
# Lo adaptamos a nuestro código fácilmente, simplificando alguna cosa
# el valor inicial x0 tiene que ser mayor que el max(x) para que podamos funcionar con la función verosimilitud_log

def constraint(parameter1, x):
    return parameter1 - np.max(x)

# minimize the negative log-likelihood subject to the constraint using the SLSQP method
x0 = 50000
mle_model = opt.minimize(verosimilitud_log, np.array(x0), method='SLSQP', 
                  constraints=[{'type': 'ineq', 'fun': constraint, 'args': (x,)}])

# cogemos los coefs estimados
param1 = mle_model.x[0]

print(mle_model)
print('Parámetros estimados por MV:\n'
      f'  - parámetro 1: {param1}')

print('\n... F U N C I O N A ! ! !')